# **Project Name – Netflix Movies and TV Shows Clustering and Content-Based Recommendation**

##### **Project Type** – Exploratory Data Analysis (EDA) / Unsupervised Machine Learning / Text Clustering
##### **Contribution** – Individual
##### **Team Member 1 –** Mohammed Shoaib
##### **Team Member 2 –** N/A
##### **Team Member 3 –** N/A
##### **Team Member 4 –** N/A

# **Project Summary -**

This project explores a Netflix titles dataset containing movies and TV shows and develops a content-based clustering workflow. The analysis begins by inspecting the dataset's structure, column types, duplicate records, and missing values. Data cleaning is applied carefully so that useful titles are retained while exact duplicate rows and inconsistent text formatting are addressed. Available metadata such as title, director, cast, country, genre/category, and description is combined into a text field for content analysis.

Exploratory data analysis is used to understand the composition of the catalogue. Visualizations examine the balance between movies and TV shows, release-year patterns, content ratings, leading countries, directors, genres and cast members, duration values, and the language used in descriptions. These charts can help describe catalogue composition and identify areas for further investigation. Exact findings should be written only after the cells have been executed and the chart outputs reviewed.

For the machine-learning component, textual metadata is normalized and transformed into numerical TF-IDF features. TF-IDF gives greater weight to terms that are informative within a title's metadata and lower weight to terms that occur very frequently. Truncated Singular Value Decomposition (SVD) reduces the feature space for exploratory visualization and for methods that work better with dense features. K-Means, a tuned K-Means model, MiniBatch K-Means, and sample-based Agglomerative Clustering are compared using the silhouette score. Since this is an unsupervised problem without known ground-truth cluster labels, classification metrics such as accuracy are not appropriate.

The resulting clusters can be inspected to determine whether titles with similar metadata are grouped together. A TF-IDF and cosine-similarity workflow can also support content-based recommendations by returning titles with text descriptions most similar to a selected title. This is a metadata-based prototype; it does not measure actual viewing behaviour, popularity, user preferences, or Netflix's internal recommendation performance.

The project demonstrates a complete analytical workflow: data quality checks, EDA, statistical hypothesis testing, text preprocessing, feature extraction, dimensionality reduction, unsupervised clustering, evaluation, and model persistence. Its business value is exploratory: it can help catalogue teams investigate content groupings and can serve as a starting point for a recommendation prototype. Results and limitations must be clearly documented, and any business decisions would require validation with user engagement and viewing data.

# **GitHub Link -**

Add the URL of your GitHub repository after uploading the final notebook and any supporting README or files.

# **Problem Statement**


The goal is to analyze the Netflix Movies and TV Shows catalogue, identify patterns in the available metadata, and group titles with similar content using unsupervised machine learning. The project also explores whether a content-based similarity method can recommend titles with similar metadata. The work includes data cleaning, exploratory visualization, hypothesis testing, text preprocessing, TF-IDF feature extraction, clustering model comparison, and interpretation of the resulting groups. Because the dataset does not contain user-level viewing histories or preference labels, the model should be treated as a metadata-based clustering/recommendation prototype rather than a production recommendation system.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# importing wordcloud library for creating word clouds
from wordcloud import WordCloud,STOPWORDS

#the Libraries below are used for text data preprocessing
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
import string
import re
string.punctuation
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

# Libraries used for Clusters Implementation
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.cluster import AgglomerativeClustering
import  scipy.cluster.hierarchy as sch

# Libarry used for building recommendation system
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Importing Warning library
import warnings
warnings.filterwarnings('ignore')


from google.colab import files



### Dataset Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load Dataset
# Confirm that the CSV path matches the file in your Google Drive.
csv_path = '/content/drive/MyDrive/NETFLIX MOVIES AND TV SHOWS CLUSTERING.csv'
try:
    df = pd.read_csv(csv_path)
    print("Loaded dataset from:", csv_path)
except FileNotFoundError:
    print("CSV was not found at the Drive path. Upload the CSV file below.")
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError("No CSV file was uploaded. Please upload the Netflix dataset CSV.")
    df = pd.read_csv(csv_files[0])
    print("Loaded uploaded dataset:", csv_files[0])


### Dataset First View

In [ ]:
# Dataset First Look
df.head()

In [ ]:
df.tail()


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print('dataset shape:',df.shape)

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
df.duplicated().sum()

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
df.isnull().sum()

In [ ]:
# Visualizing the missing values
df.isnull().sum().plot(kind='bar', figsize=(10,4))
plt.title("Missing values in Each Column")
plt.xlabel("Columns")
plt.ylabel("Number Of Missing values")
plt.xticks(rotation=45)
plt.show()

### What did you know about your dataset?

The dataset contains title-level records for Netflix movies and TV shows. The initial checks establish its row and column counts, data types, duplicate count, and missing-value profile. The exact figures should be taken from the outputs of the notebook after running it on the supplied CSV. Fields such as title, type, director, cast, country, date added, release year, rating, duration, listed-in categories, and description are commonly present in this dataset; the code checks the actual columns before using optional fields. Missing metadata is expected in some columns, so it is handled according to the analysis rather than deleting every incomplete record.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset columns
print("Columns:", df.columns.tolist())


In [ ]:
# Dataset descriptive statistics
display(df.describe(include="all").T)


### Variables Description

- `show_id`: identifier for a catalogue record, if present.
- `type`: whether the title is a Movie or TV Show.
- `title`: title name.
- `director`: director metadata; may contain multiple names.
- `cast`: cast metadata; may contain multiple names.
- `country`: country/countries associated with the title.
- `date_added`: date the title was added to the catalogue.
- `release_year`: year the title was released.
- `rating`: content rating classification.
- `duration`: movie runtime in minutes or TV show season count.
- `listed_in`: genre/category labels.
- `description`: short text description of the title.

The CSV may have a different set of columns; `df.columns` is the source of truth. `duration_value` and `date_added_year` are derived features used for visualization. `content_text`, `clean_text`, `tokens`, and `normalized_text` are derived text-processing fields.

### Check Unique Values for each variable.

In [ ]:
# Number of unique values in each column
unique_counts = df.nunique(dropna=False).sort_values(ascending=False)
display(unique_counts.to_frame("unique_values"))


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Data wrangling: preserve raw data and prepare analysis columns
df = df.copy()
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Standard Netflix dataset columns are expected; this check gives a helpful error if the CSV differs.
required = {"type", "title"}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {missing_required}. Available columns: {df.columns.tolist()}")

# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Clean text-like columns without inventing values
text_cols = [c for c in ["title", "director", "cast", "country", "rating", "duration", "listed_in", "description"] if c in df.columns]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

# Convert date_added and release_year when available
if "date_added" in df.columns:
    df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
if "release_year" in df.columns:
    df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")

# Derived fields used in EDA and modeling
df["type"] = df["type"].str.title()
if "duration" in df.columns:
    df["duration_value"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)")[0], errors="coerce")
if "date_added" in df.columns:
    df["date_added_year"] = df["date_added"].dt.year

# A combined text field is useful for content-based clustering/recommendations.
text_features = [c for c in ["title", "director", "cast", "country", "listed_in", "description"] if c in df.columns]
df["content_text"] = df[text_features].fillna("").astype(str).agg(" ".join, axis=1).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

print("Prepared dataset shape:", df.shape)
display(df.head())


### What all manipulations have you done and insights you found?

The data-wrangling stage standardizes column names and text whitespace, removes exact duplicate rows, converts date and year columns to suitable types, and creates derived fields for duration, date-added year, and combined text metadata. Missing values in descriptive fields are not treated as proof that a title is invalid; they are handled as empty text for the content-based model. The combined text field is used for clustering so that the model can compare available title metadata. Check the printed shape and preview to confirm the transformations ran successfully.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart 1: Count of Movies and TV Shows
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="type", order=df["type"].value_counts().index)
plt.title("Number of Movies vs TV Shows")
plt.xlabel("Content type"); plt.ylabel("Number of titles")
plt.tight_layout(); plt.show()


##### 1. Why did you pick the specific chart?

A count plot makes the catalogue composition easy to compare by content type.

##### 2. What is/are the insight(s) found from the chart?

After running the chart, report which type has more records and the size of the difference; do not infer audience watch time from catalogue counts alone.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The result can help describe catalogue balance and identify whether further analysis should focus on movies or TV shows. Catalogue volume is not the same as demand.

#### Chart - 2

In [ ]:
# Chart 2: Share of Movies and TV Shows
type_counts = df["type"].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(type_counts.values, labels=type_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Netflix Content Mix")
plt.tight_layout(); plt.show()


##### 1. Why did you pick the specific chart?

A pie chart communicates each content type's share of all listed titles.

##### 2. What is/are the insight(s) found from the chart?

Use the displayed percentages to state the share of Movies and TV Shows. The chart reflects the dataset, not necessarily the complete current Netflix catalogue.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The proportions can support catalogue planning, but should be combined with engagement and licensing data before business decisions.

#### Chart - 3

In [ ]:
# Chart 3: Titles released by year
if "release_year" in df.columns:
    yearly = df.dropna(subset=["release_year"]).groupby("release_year").size().sort_index()
    plt.figure(figsize=(12, 4))
    plt.plot(yearly.index, yearly.values)
    plt.title("Number of Netflix Titles by Release Year")
    plt.xlabel("Release year"); plt.ylabel("Number of titles")
    plt.tight_layout(); plt.show()
else:
    print("The dataset has no release_year column.")


##### 1. Why did you pick the specific chart?

A line chart shows how the number of titles varies across release years and highlights long-term patterns.

##### 2. What is/are the insight(s) found from the chart?

Describe the visible peaks and dips after running the cell. Recent years may have incomplete representation depending on how the dataset was collected.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Release-year patterns can inform catalogue-age analysis and content acquisition research, but do not by themselves prove demand.

#### Chart - 4

In [ ]:
# Chart 4: Content rating distribution
if "rating" in df.columns:
    top_ratings = df["rating"].fillna("Unknown").value_counts().head(12)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=top_ratings.values, y=top_ratings.index)
    plt.title("Most Common Content Ratings")
    plt.xlabel("Number of titles"); plt.ylabel("Rating")
    plt.tight_layout(); plt.show()


##### 1. Why did you pick the specific chart?

A bar chart compares the frequency of rating categories and is easier to rank than a pie chart with many categories.

##### 2. What is/are the insight(s) found from the chart?

Identify the most common ratings from the chart output and mention whether missing/unknown ratings are substantial.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Rating distribution can help teams understand the catalogue's maturity profile and content mix. It does not directly indicate audience preference.

#### Chart - 5

In [ ]:
# Chart 5: Top 10 countries by number of titles
if "country" in df.columns:
    country_counts = (df["country"].dropna().str.split(", ").explode().str.strip()
                      .replace("", np.nan).dropna().value_counts().head(10))
    plt.figure(figsize=(10, 5))
    sns.barplot(x=country_counts.values, y=country_counts.index)
    plt.title("Top 10 Countries by Netflix Titles")
    plt.xlabel("Number of titles"); plt.ylabel("Country")
    plt.tight_layout(); plt.show()


##### 1. Why did you pick the specific chart?

A horizontal bar chart makes country names readable while comparing the ten most frequent country labels.

##### 2. What is/are the insight(s) found from the chart?

State the leading countries shown in the executed chart. Multi-country records are split into individual country labels, so totals can exceed the number of titles.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Country coverage can reveal geographic concentration and potential gaps, but country metadata may be missing or inconsistently recorded.

#### Chart - 6

In [ ]:
# Chart 6: Top 10 directors by number of titles
if "director" in df.columns:
    director_counts = (df["director"].dropna().str.split(", ").explode().str.strip()
                       .replace("", np.nan).dropna().value_counts().head(10))
    plt.figure(figsize=(10, 5))
    sns.barplot(x=director_counts.values, y=director_counts.index)
    plt.title("Top 10 Directors by Number of Titles")
    plt.xlabel("Number of titles"); plt.ylabel("Director")
    plt.tight_layout(); plt.show()


##### 1. Why did you pick the specific chart?

A horizontal bar chart clearly ranks directors by the number of titles credited in the dataset.

##### 2. What is/are the insight(s) found from the chart?

Use the actual chart to name the leading directors. Records with multiple directors are split, so a title may contribute to more than one director.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Director counts can help explore catalogue depth and recurring creative partnerships; they do not measure quality or popularity.

#### Chart - 7

In [ ]:
# Chart 7: Top 10 genres/categories
if "listed_in" in df.columns:
    genre_counts = (df["listed_in"].dropna().str.split(", ").explode().str.strip()
                    .replace("", np.nan).dropna().value_counts().head(10))
    plt.figure(figsize=(10, 5))
    sns.barplot(x=genre_counts.values, y=genre_counts.index)
    plt.title("Top 10 Netflix Genres/Categories")
    plt.xlabel("Number of titles"); plt.ylabel("Genre")
    plt.tight_layout(); plt.show()


##### 1. Why did you pick the specific chart?

Genre labels are categorical and a ranked bar chart clearly shows which categories occur most often.

##### 2. What is/are the insight(s) found from the chart?

Name the most frequent categories after running the chart. Each title can have multiple categories, so category counts are not mutually exclusive.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Genre distribution can guide further content-mix analysis and recommendation exploration, but frequency is not equivalent to viewer demand.

#### Chart - 8

In [ ]:
# Chart 8: Top 10 cast members
if "cast" in df.columns:
    cast_counts = (df["cast"].dropna().str.split(", ").explode().str.strip()
                   .replace("", np.nan).dropna().value_counts().head(10))
    plt.figure(figsize=(10, 5))
    sns.barplot(x=cast_counts.values, y=cast_counts.index)
    plt.title("Top 10 Cast Members by Number of Credits")
    plt.xlabel("Number of credits"); plt.ylabel("Cast member")
    plt.tight_layout(); plt.show()


##### 1. Why did you pick the specific chart?

A ranked bar chart makes the most frequently credited cast members easy to compare.

##### 2. What is/are the insight(s) found from the chart?

Use the chart output to identify the top cast members. Titles with multiple cast names contribute to multiple counts.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Cast metadata may help create content discovery features, but credit frequency alone should not be interpreted as popularity or performance.

#### Chart - 9

In [ ]:
# Chart 9: Duration value distribution
if "duration_value" not in df.columns and "duration" in df.columns:
    df["duration_value"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)")[0], errors="coerce")
if "duration_value" in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x="duration_value", hue="type", bins=30, element="step", stat="count", common_norm=False)
    plt.title("Distribution of Duration Values by Content Type")
    plt.xlabel("Minutes for movies / seasons for TV shows"); plt.ylabel("Number of titles")
    plt.tight_layout(); plt.show()


##### 1. Why did you pick the specific chart?

A histogram shows the spread of duration values and can reveal common runtimes or season counts.

##### 2. What is/are the insight(s) found from the chart?

Interpret Movies and TV Shows separately: movie duration is usually in minutes, while TV Show duration is usually seasons. Do not compare their raw values as the same unit.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Duration can support format-level catalogue analysis, but the two content types use different units.

#### Chart - 10

In [ ]:
# Chart 10: Release-year distribution by content type
if "release_year" in df.columns:
    recent = df.dropna(subset=["release_year"]).query("release_year >= 1980")
    plt.figure(figsize=(10, 5))
    sns.histplot(data=recent, x="release_year", hue="type", bins=25, multiple="stack")
    plt.title("Release-Year Distribution by Content Type")
    plt.xlabel("Release year"); plt.ylabel("Number of titles")
    plt.tight_layout(); plt.show()


##### 1. Why did you pick the specific chart?

A stacked histogram compares the release-year distributions of Movies and TV Shows in one view.

##### 2. What is/are the insight(s) found from the chart?

Describe which periods show relatively more titles of each type. The code focuses on 1980 onward for readability.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This can identify differences in the historical mix of formats and suggest follow-up analyses by genre or country.

#### Chart - 11

In [ ]:
# Chart 11: Titles added to Netflix by year
if "date_added" in df.columns:
    added = df.dropna(subset=["date_added"]).groupby(df["date_added"].dt.year).size()
    plt.figure(figsize=(9, 4))
    sns.lineplot(x=added.index, y=added.values, marker="o")
    plt.title("Titles Added to Netflix by Year")
    plt.xlabel("Year added"); plt.ylabel("Number of titles")
    plt.tight_layout(); plt.show()


##### 1. Why did you pick the specific chart?

A line chart shows how the number of catalogue additions changes over time.

##### 2. What is/are the insight(s) found from the chart?

Describe peaks and dips based on the chart. The dataset's coverage window and incomplete dates can affect the pattern.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Addition patterns can support operational and catalogue planning analysis, but do not establish why titles were added or removed.

#### Chart - 12

In [ ]:
# Chart 12: Word cloud of descriptions
if "description" in df.columns:
    corpus = " ".join(df["description"].dropna().astype(str).tolist())
    if corpus.strip():
        wc = WordCloud(width=1000, height=450, background_color="white", stopwords=STOPWORDS,
                       collocations=False).generate(corpus)
        plt.figure(figsize=(14, 6)); plt.imshow(wc, interpolation="bilinear")
        plt.axis("off"); plt.title("Frequently Used Words in Netflix Descriptions")
        plt.show()
    else:
        print("No descriptions available for word cloud.")


##### 1. Why did you pick the specific chart?

A word cloud gives a quick visual summary of frequently occurring terms in title descriptions.

##### 2. What is/are the insight(s) found from the chart?

Mention prominent words that are actually visible in your output. Word clouds show frequency, not sentiment, importance, or causal themes.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

It can help generate hypotheses about catalogue themes and guide deeper text analysis, but should not be used as the only evidence.

#### Chart - 13

In [ ]:
# Chart 13: Average duration by content type (duration values are not directly comparable)
if "duration_value" in df.columns:
    duration_summary = df.groupby("type")["duration_value"].median().dropna().sort_values()
    plt.figure(figsize=(7, 4))
    sns.barplot(x=duration_summary.index, y=duration_summary.values)
    plt.title("Median Duration Value by Content Type")
    plt.xlabel("Content type"); plt.ylabel("Median duration value")
    plt.tight_layout(); plt.show()
    print("Note: movie duration is in minutes, while TV duration is in seasons; compare within type rather than directly.")


##### 1. Why did you pick the specific chart?

A bar chart summarizes the median numeric duration value for each type.

##### 2. What is/are the insight(s) found from the chart?

Movie duration is measured in minutes and TV Show duration in seasons, so the numeric values are not directly comparable. Interpret within each type.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The summary can describe typical formats within each type; it should not be presented as a direct movie-versus-show length comparison.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Chart 14: Correlation heatmap for numeric columns
numeric_df = df.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    plt.figure(figsize=(9, 6))
    sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
    plt.title("Correlation Heatmap of Numeric Features")
    plt.tight_layout(); plt.show()
else:
    print("At least two numeric columns are required for a correlation heatmap.")


A correlation heatmap summarizes linear correlations between numeric fields. It is useful for quickly identifying strongly related numeric variables, although correlation does not imply causation.

A correlation heatmap summarizes linear correlations between numeric fields. It is useful for quickly identifying strongly related numeric variables, although correlation does not imply causation.

Use the displayed coefficients to report any meaningful relationships. With this dataset, numeric fields may include release year, duration value, and date-added year. Interpret duration cautiously because its units differ by content type.

Use the displayed coefficients to report any meaningful relationships. Numeric fields may include release year, duration value, and date-added year. Interpret duration cautiously because its units differ by content type.

#### Chart - 15 - Pair Plot

In [ ]:
# Chart 15: Pair plot of selected numeric features (sampled for performance)
pair_cols = [c for c in ["release_year", "duration_value", "date_added_year"] if c in df.columns]
pair_data = df[pair_cols].dropna()
if len(pair_cols) >= 2 and not pair_data.empty:
    pair_data = pair_data.sample(min(1000, len(pair_data)), random_state=42)
    sns.pairplot(pair_data, corner=True, diag_kind="hist")
    plt.suptitle("Pair Plot of Numeric Features", y=1.02)
    plt.show()
else:
    print("Not enough numeric features for a pair plot.")


A pair plot shows pairwise distributions and relationships among a small set of numeric features. Sampling limits plotting time and improves readability.

A pair plot shows pairwise distributions and relationships among a small set of numeric features. Sampling limits plotting time and improves readability.

Describe any visible clusters, skew, or relationships from the generated plot. The plot is exploratory; do not claim causation from visual correlation.

Describe any visible clusters, skew, or relationships from the generated plot. The plot is exploratory; do not claim causation from visual correlation.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Three hypotheses are tested at a 5% significance level. (1) Movie and TV Show release-year distributions differ. (2) Content type and rating are associated. (3) The distribution of years in which titles were added differs by content type. These tests evaluate associations in the available catalogue data; they do not establish causation or explain why Netflix made a content decision.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

H₀: The release-year distributions for Movies and TV Shows are the same. H₁: The distributions differ. The Mann–Whitney U test is used because release year is numeric/discrete and the comparison does not require a normal distribution.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Hypothesis 1: Movie and TV show release years differ (Mann–Whitney U test)
from scipy.stats import mannwhitneyu, chi2_contingency
if "release_year" in df.columns:
    movies_year = df.loc[df["type"] == "Movie", "release_year"].dropna()
    tv_year = df.loc[df["type"] == "TV Show", "release_year"].dropna()
    if len(movies_year) and len(tv_year):
        stat, p_value = mannwhitneyu(movies_year, tv_year, alternative="two-sided")
        print(f"Movie median release year: {movies_year.median():.0f}")
        print(f"TV Show median release year: {tv_year.median():.0f}")
        print(f"Mann–Whitney U statistic={stat:.3f}, p-value={p_value:.6g}")
        print("Decision at alpha=0.05:", "Reject H0" if p_value < 0.05 else "Fail to reject H0")
    else:
        print("Insufficient release-year data for both content types.")
else:
    print("release_year column not available.")


##### Which statistical test have you done to obtain P-Value?

The Mann–Whitney U test was used to obtain the p-value.

##### Why did you choose the specific statistical test?

The Mann–Whitney U test compares two independent groups without requiring normally distributed release-year values. It is suitable as a non-parametric comparison of the Movie and TV Show release-year distributions.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

H₀: Content type and content rating are independent. H₁: Content type and content rating are associated.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Hypothesis 2: Content type and rating are associated (Chi-square test)
if {"type", "rating"}.issubset(df.columns):
    contingency = pd.crosstab(df["type"], df["rating"].fillna("Unknown"))
    if contingency.shape[0] >= 2 and contingency.shape[1] >= 2:
        chi2, p_value, dof, expected = chi2_contingency(contingency)
        print(f"Chi-square={chi2:.3f}, degrees of freedom={dof}, p-value={p_value:.6g}")
        print("Decision at alpha=0.05:", "Reject H0: type and rating are associated" if p_value < 0.05 else "Fail to reject H0: insufficient evidence of association")
    else:
        print("Insufficient categories for chi-square test.")


##### Which statistical test have you done to obtain P-Value?

The chi-square test of independence was used to obtain the p-value.

##### Why did you choose the specific statistical test?

The chi-square test is appropriate because content type and content rating are categorical variables. Check expected cell counts; sparse categories can affect the reliability of the approximation.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

H₀: The year-added distribution is independent of content type. H₁: The year-added distribution differs by content type.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Hypothesis 3: Netflix content added differs by type (Chi-square test on date-added year)
if {"type", "date_added"}.issubset(df.columns):
    tmp = df.dropna(subset=["date_added"]).copy()
    tmp["added_year"] = tmp["date_added"].dt.year
    table = pd.crosstab(tmp["type"], tmp["added_year"])
    if table.shape[0] >= 2 and table.shape[1] >= 2:
        chi2, p_value, dof, expected = chi2_contingency(table)
        print(f"Chi-square={chi2:.3f}, degrees of freedom={dof}, p-value={p_value:.6g}")
        print("Decision at alpha=0.05:", "Reject H0: addition-year distribution differs by type" if p_value < 0.05 else "Fail to reject H0")
    else:
        print("Insufficient date-added data for this test.")


##### Which statistical test have you done to obtain P-Value?

The chi-square test of independence was used to obtain the p-value.

##### Why did you choose the specific statistical test?

The data are represented as a contingency table of content type by year added, making a chi-square test of independence a reasonable starting point. Sparse year categories should be reviewed.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handle missing values for modeling without deleting useful titles
model_df = df.copy()
for col in ["title", "director", "cast", "country", "listed_in", "description", "rating", "duration"]:
    if col in model_df.columns:
        model_df[col] = model_df[col].fillna("")
model_df["content_text"] = model_df["content_text"].fillna("")
print("Missing values in text fields after filling:")
display(model_df[[c for c in ["title","director","cast","country","listed_in","description"] if c in model_df.columns]].isna().sum())


#### What all missing value imputation techniques have you used and why did you use those techniques?

For text modeling, missing director, cast, country, genre, rating, duration, and description values are replaced with empty strings. This preserves titles with incomplete metadata and avoids introducing artificial words such as 'nan'. Missing dates and numeric values remain missing where appropriate for statistical summaries. Imputation is chosen according to each column's role rather than using one method for every variable.

### 2. Handling Outliers

In [ ]:
# Check numeric outliers; do not remove release years/duration automatically because extremes may be valid
numeric_cols = model_df.select_dtypes(include=np.number).columns.tolist()
if numeric_cols:
    display(model_df[numeric_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T)
    for col in numeric_cols:
        q1, q3 = model_df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        if pd.notna(iqr) and iqr > 0:
            outlier_count = ((model_df[col] < q1 - 1.5*iqr) | (model_df[col] > q3 + 1.5*iqr)).sum()
            print(f"{col}: {outlier_count} potential IQR outliers (reviewed, not automatically removed)")
else:
    print("No numeric columns available for outlier inspection.")


##### What all outlier treatment techniques have you used and why did you use those techniques?

Potential numeric outliers are flagged with the IQR rule and reviewed rather than automatically deleted. Extreme release years or durations may represent valid catalogue records, and the duration column mixes minutes for movies with seasons for TV shows. Removing such rows without domain review could discard useful information.

### 3. Categorical Encoding

In [ ]:
# Encode categorical metadata only for analysis where needed; TF-IDF below handles text representation
categorical_cols = [c for c in ["type", "rating"] if c in model_df.columns]
for col in categorical_cols:
    model_df[col + "_code"] = model_df[col].astype("category").cat.codes
display(model_df[[c for c in categorical_cols] + [c + "_code" for c in categorical_cols]].head())


#### What all categorical encoding techniques have you used & why did you use those techniques?

Categorical codes are created for `type` and `rating` for compact exploratory use. These integer codes are not treated as ordered numeric measurements. The clustering model instead uses TF-IDF text vectors, which better represent the words and metadata in each title.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand common English contractions (lightweight dictionary; applied to combined text)
contractions = {"can't":"cannot", "won't":"will not", "n't":" not", "i'm":"i am",
                "it's":"it is", "that's":"that is", "what's":"what is", "you're":"you are",
                "we're":"we are", "they're":"they are", "i've":"i have", "we've":"we have",
                "doesn't":"does not", "don't":"do not", "isn't":"is not", "aren't":"are not"}
def expand_contractions(text):
    text = str(text)
    for old, new in contractions.items():
        text = re.sub(re.escape(old), new, text, flags=re.IGNORECASE)
    return text
model_df["clean_text"] = model_df["content_text"].map(expand_contractions)


#### 2. Lower Casing

In [ ]:
# Lowercase text
model_df["clean_text"] = model_df["clean_text"].str.lower()


#### 3. Removing Punctuations

In [ ]:
# Remove punctuation
model_df["clean_text"] = model_df["clean_text"].str.translate(str.maketrans("", "", string.punctuation))


#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs and tokens containing digits
model_df["clean_text"] = model_df["clean_text"].str.replace(r"https?\S+|www\.\S+", " ", regex=True)
model_df["clean_text"] = model_df["clean_text"].str.replace(r"\b\w*\d\w*\b", " ", regex=True)


#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove English stopwords while retaining content-bearing terms
stop_words = set(stopwords.words("english"))
def remove_stopwords(text):
    return " ".join(word for word in str(text).split() if word not in stop_words)
model_df["clean_text"] = model_df["clean_text"].map(remove_stopwords)


In [ ]:
# Normalize extra whitespace
model_df["clean_text"] = model_df["clean_text"].str.replace(r"\s+", " ", regex=True).str.strip()
display(model_df[["content_text", "clean_text"]].head())


#### 6. Rephrase Text

In [ ]:
# Rephrase text: no paraphrasing is applied because it could alter original title/plot meaning.
# Instead, normalize repeated whitespace and missing text safely.
model_df["clean_text"] = model_df["clean_text"].fillna("").str.strip()


#### 7. Tokenization

In [ ]:
# Tokenization: split cleaned text into word tokens
model_df["tokens"] = model_df["clean_text"].map(lambda x: x.split())
display(model_df[["clean_text", "tokens"]].head())


#### 8. Text Normalization

In [ ]:
# Text normalization using Snowball stemming
stemmer = SnowballStemmer("english")
model_df["normalized_text"] = model_df["tokens"].map(lambda tokens: " ".join(stemmer.stem(token) for token in tokens))
display(model_df[["clean_text", "normalized_text"]].head())


##### Which text normalization technique have you used and why?

The text is lowercased, punctuation, URLs, digit-containing tokens, and common English stopwords are removed, and whitespace is normalized. Snowball stemming reduces related word forms to a shared root. This can reduce vocabulary size, although stemming may make words less readable; the original metadata remains available for interpretation.

#### 9. Part of speech tagging

In [ ]:
# POS tagging is optional for this clustering task; download tagger if you specifically need POS labels.
# It is not required for TF-IDF clustering, so we avoid adding runtime cost to every title.
print("POS tagging skipped: TF-IDF content clustering does not require part-of-speech labels.")


#### 10. Text Vectorization

In [ ]:
# Convert text to numeric TF-IDF features
tfidf = TfidfVectorizer(max_features=20000, min_df=2, max_df=0.95, ngram_range=(1, 2), sublinear_tf=True)
X_tfidf = tfidf.fit_transform(model_df["normalized_text"].fillna(""))
print("TF-IDF matrix shape:", X_tfidf.shape)


##### Which text vectorization technique have you used and why?

TF-IDF vectorization is used because it converts text into a sparse numeric representation while reducing the influence of terms that occur in almost every record. Unigrams and bigrams capture individual terms and short phrases. A maximum vocabulary size and document-frequency thresholds control memory use and reduce extremely rare or common terms.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Feature manipulation: select useful metadata text and create a combined normalized field
# content_text was created from title, director, cast, country, genres and description.
# This creates a content-based representation rather than numeric prediction features.
print("Text source columns:", [c for c in ["title","director","cast","country","listed_in","description"] if c in model_df.columns])
print("Non-empty text records:", int((model_df["normalized_text"].str.len() > 0).sum()))


#### 2. Feature Selection

In [ ]:
# Feature selection: TF-IDF limits vocabulary size and removes very rare/common terms
feature_names = np.array(tfidf.get_feature_names_out())
print(f"Selected {len(feature_names):,} TF-IDF terms.")


Feature selection is performed through TF-IDF vocabulary limits (`max_features`, `min_df`, and `max_df`) and by selecting content-related metadata fields. This reduces noise and memory use. Feature importance in unsupervised clustering is not equivalent to feature importance in a supervised predictive model; cluster keywords should be inspected to understand what differentiates groups.

Feature selection is performed through TF-IDF vocabulary limits (`max_features`, `min_df`, and `max_df`) and by selecting content-related metadata fields. This reduces noise and memory use. Feature importance in unsupervised clustering is not equivalent to feature importance in a supervised predictive model; cluster keywords should be inspected to understand what differentiates groups.

##### Which all features you found important and why?

The most useful features are expected to be informative words and phrases from titles, genres, cast, directors, countries, and descriptions. The actual discriminating terms should be inspected from cluster-level TF-IDF summaries before making a claim about which features are most important.

### 5. Data Transformation

Yes. Text is transformed into TF-IDF vectors because clustering algorithms require numerical features. Truncated SVD further reduces the sparse text representation to a dense lower-dimensional representation for visualization and hierarchical clustering.

In [ ]:
# Text data is transformed into TF-IDF vectors in cell 226. Reduce sparse vectors for visualization/alternative clustering.
from sklearn.decomposition import TruncatedSVD
n_components = min(100, max(2, X_tfidf.shape[1] - 1))
svd = TruncatedSVD(n_components=n_components, random_state=42)
X_reduced = svd.fit_transform(X_tfidf)
print("Reduced feature matrix shape:", X_reduced.shape)
print(f"Variance captured by {n_components} SVD components: {svd.explained_variance_ratio_.sum():.3f}")


### 6. Data Scaling

In [ ]:
# Scale reduced dense features for algorithms sensitive to scale
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reduced)
print("Scaled feature matrix shape:", X_scaled.shape)


TF-IDF itself provides feature weighting for text. StandardScaler is then applied to the SVD-reduced dense features for methods that use distance calculations. Scaling is applied to the reduced representation, not to the original categorical codes as if they were continuous quantities.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Yes. The combined text field can create a high-dimensional sparse matrix. Truncated SVD reduces it to a smaller dense feature space, which improves visualization and makes sample-based hierarchical clustering more practical. The retained explained variance is printed by the notebook.

In [ ]:
# Dimensionality reduction is useful because text TF-IDF can have thousands of dimensions.
# TruncatedSVD was applied above; display first two components for a quick view.
if X_reduced.shape[1] >= 2:
    plt.figure(figsize=(8, 5))
    plt.scatter(X_reduced[:, 0], X_reduced[:, 1], s=8, alpha=0.5)
    plt.title("Netflix Titles: First Two SVD Components")
    plt.xlabel("SVD component 1"); plt.ylabel("SVD component 2")
    plt.tight_layout(); plt.show()


##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

Truncated SVD is used because it can reduce sparse TF-IDF matrices without first converting the entire matrix into a dense array. It is a practical dimensionality-reduction choice for text data.

### 8. Data Splitting

In [ ]:
# This is an unsupervised learning task, so no train/test split is required for clustering.
# Use a reproducible sample for model selection metrics to keep runtime manageable.
rng = np.random.RandomState(42)
sample_size = min(2000, X_tfidf.shape[0])
sample_idx = rng.choice(X_tfidf.shape[0], size=sample_size, replace=False)
X_eval = X_tfidf[sample_idx]
X_eval_reduced = X_scaled[sample_idx]
print(f"Clustering is unsupervised; evaluation sample size: {sample_size}")


##### What data splitting ratio have you used and why?

A conventional supervised train/test split is not required because the goal is to discover groups rather than predict known labels. A reproducible sample is used for model-selection metrics to keep computation manageable. For a production evaluation, stability across samples and human review of cluster quality would also be useful.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Class imbalance is not applicable before clustering because the dataset has no target class label. After fitting, cluster sizes are checked to see whether the algorithm assigns most titles to one group or produces very small groups.

In [ ]:
# Class imbalance is not applicable in unsupervised clustering because there is no target label.
# After clustering, inspect cluster sizes to check whether one cluster dominates.
print("No target class exists; class-imbalance resampling is not applicable.")


##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

No resampling method such as SMOTE is used because there is no ground-truth class label to balance. If the resulting clusters are highly uneven, the model settings and data representation should be reviewed instead of artificially balancing records.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model 1: K-Means clustering on TF-IDF features
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_tfidf)
model_df["kmeans_cluster"] = cluster_labels
eval_labels = cluster_labels[sample_idx]
sil = silhouette_score(X_eval, eval_labels) if len(set(eval_labels)) > 1 else np.nan
print(f"K-Means silhouette score (sample): {sil:.4f}")
display(model_df["kmeans_cluster"].value_counts().sort_index().to_frame("title_count"))


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# K-Means evaluation metric chart
cluster_sizes = model_df["kmeans_cluster"].value_counts().sort_index()
plt.figure(figsize=(8, 4)); sns.barplot(x=cluster_sizes.index.astype(str), y=cluster_sizes.values)
plt.title("K-Means Cluster Sizes"); plt.xlabel("Cluster"); plt.ylabel("Number of titles")
plt.tight_layout(); plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Model 1 tuning: compare a small range of K values using silhouette score on a sample
k_results = []
for k in [3, 4, 5, 6, 7, 8]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_eval)
    score = silhouette_score(X_eval, labels) if len(set(labels)) > 1 else np.nan
    k_results.append({"k": k, "silhouette": score, "inertia": km.inertia_})
k_results_df = pd.DataFrame(k_results)
display(k_results_df)
best_k = int(k_results_df.sort_values("silhouette", ascending=False).iloc[0]["k"])
best_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
best_labels = best_kmeans.fit_predict(X_tfidf)
model_df["best_kmeans_cluster"] = best_labels
print("Best k by sampled silhouette score:", best_k)


##### Which hyperparameter optimization technique have you used and why?

K-Means partitions the TF-IDF vectors into k groups by minimizing within-cluster squared distances. The initial model uses six clusters as a baseline. The silhouette score estimates how well separated clusters are; higher values generally indicate better separation, but text clusters should also be reviewed for meaningful content.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The notebook compares several values of k using silhouette score on a reproducible sample, then refits K-Means using the best sampled score. Use the printed score table and chart to report the selected k. A higher silhouette score is not sufficient by itself; inspect cluster contents for interpretability.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Compare K-Means choices
plt.figure(figsize=(8, 4))
sns.lineplot(data=k_results_df, x="k", y="silhouette", marker="o")
plt.title("K-Means Hyperparameter Selection")
plt.xlabel("Number of clusters (k)"); plt.ylabel("Silhouette score")
plt.tight_layout(); plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model 2: MiniBatch K-Means (faster alternative for larger datasets)
from sklearn.cluster import MiniBatchKMeans
mbk = MiniBatchKMeans(n_clusters=best_k, random_state=42, batch_size=1024, n_init=10)
mb_labels = mbk.fit_predict(X_tfidf)
model_df["minibatch_cluster"] = mb_labels
mb_eval = mb_labels[sample_idx]
mb_sil = silhouette_score(X_eval, mb_eval) if len(set(mb_eval)) > 1 else np.nan
print(f"MiniBatch K-Means silhouette score (sample): {mb_sil:.4f}")


##### Which hyperparameter optimization technique have you used and why?

MiniBatch K-Means updates cluster centers using small batches, which can reduce computation on larger datasets. It is compared with standard K-Means using the same sampled silhouette metric.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Compare the reported silhouette score with the tuned K-Means score. Small differences may reflect the approximation used by MiniBatch K-Means. Select a model based on both clustering quality and runtime, and review actual title examples in each cluster.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

The silhouette score measures separation and cohesion: values closer to 1 indicate better-defined clusters, values around 0 indicate overlapping clusters, and negative values may indicate titles assigned to the wrong group. Cluster sizes indicate whether the catalogue is divided evenly. These metrics can help evaluate a content grouping prototype, but they do not measure user satisfaction, watch time, or recommendation click-through rate.

### ML Model - 3

In [ ]:
# ML Model 3: Agglomerative clustering on a bounded sample of reduced features
from sklearn.cluster import AgglomerativeClustering
agg_size = min(1500, X_scaled.shape[0])
agg_idx = np.random.RandomState(42).choice(X_scaled.shape[0], size=agg_size, replace=False)
agg_model = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
agg_labels = agg_model.fit_predict(X_scaled[agg_idx])
agg_sil = silhouette_score(X_scaled[agg_idx], agg_labels) if len(set(agg_labels)) > 1 else np.nan
print(f"Agglomerative silhouette score (sample of {agg_size}): {agg_sil:.4f}")


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Compare the three clustering approaches with silhouette scores
comparison = pd.DataFrame({
    "Model": ["K-Means (initial k=6)", f"K-Means (tuned k={best_k})", f"MiniBatch K-Means (k={best_k})", f"Agglomerative (sample, k={best_k})"],
    "Silhouette score": [sil, silhouette_score(X_eval, best_labels[sample_idx]) if len(set(best_labels[sample_idx])) > 1 else np.nan,
                         mb_sil, agg_sil]
})
display(comparison)
plt.figure(figsize=(10, 4))
sns.barplot(data=comparison, x="Model", y="Silhouette score")
plt.xticks(rotation=20, ha="right"); plt.title("Clustering Model Comparison")
plt.tight_layout(); plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Agglomerative clustering has no simple K-Means-style centroid tuning.
# Compare linkage choices on a smaller sample, because hierarchical clustering is computationally expensive.
linkage_results = []
for linkage in ["ward", "average", "complete"]:
    try:
        agg = AgglomerativeClustering(n_clusters=best_k, linkage=linkage)
        labels = agg.fit_predict(X_scaled[agg_idx])
        score = silhouette_score(X_scaled[agg_idx], labels) if len(set(labels)) > 1 else np.nan
        linkage_results.append({"linkage": linkage, "silhouette": score})
    except Exception as e:
        linkage_results.append({"linkage": linkage, "silhouette": np.nan, "error": str(e)})
linkage_df = pd.DataFrame(linkage_results)
display(linkage_df)


The hierarchical model is evaluated on a bounded sample because Agglomerative Clustering can be expensive on a large dataset. Linkage methods are compared using silhouette score. This is a computationally constrained comparison, not proof that one clustering method is universally best.

The Agglomerative model is compared across Ward, average, and complete linkage on a fixed sample. The selected linkage should be based on the printed silhouette scores and cluster interpretability, with runtime also considered.

Use the linkage comparison table to see whether any tested linkage improves the sampled silhouette score. The result should be interpreted alongside cluster contents and runtime; hierarchical clustering is only fit on a sample in this notebook.

Use the linkage comparison table to see whether any tested linkage improves the sampled silhouette score. The result should be interpreted alongside cluster contents and runtime; hierarchical clustering is only fit on a sample in this notebook.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Silhouette score is the primary numerical metric because the task has no ground-truth labels. Cluster size distribution and qualitative review of example titles are also important. A real business evaluation would require an offline relevance study or online A/B test using user-level engagement data.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

Choose the model with the best combination of silhouette score, stable cluster sizes, interpretable title groupings, and reasonable runtime. The notebook saves the tuned K-Means model as the practical baseline; if another model is better on the metrics and inspection, update the final choice rather than claiming K-Means always wins.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

K-Means is an unsupervised model and does not provide standard supervised feature importance. To explain clusters, inspect the highest-weight TF-IDF terms and example titles in each cluster. The following optional code cell can be used to print representative terms and titles for each cluster.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the TF-IDF vectorizer and tuned K-Means model for reuse
import joblib, os
os.makedirs("netflix_clustering_artifacts", exist_ok=True)
joblib.dump(tfidf, "netflix_clustering_artifacts/tfidf_vectorizer.joblib")
joblib.dump(best_kmeans, "netflix_clustering_artifacts/kmeans_model.joblib")
joblib.dump(svd, "netflix_clustering_artifacts/svd_reducer.joblib")
print("Saved model artifacts in ./netflix_clustering_artifacts/")


### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Reload saved artifacts and assign clusters to unseen sample titles as a sanity check
loaded_tfidf = joblib.load("netflix_clustering_artifacts/tfidf_vectorizer.joblib")
loaded_model = joblib.load("netflix_clustering_artifacts/kmeans_model.joblib")
unseen_texts = [
    "mystery crime investigation detective thriller",
    "romantic comedy love relationship wedding",
    "nature documentary wildlife ocean planet"
]
unseen_matrix = loaded_tfidf.transform(unseen_texts)
predicted_clusters = loaded_model.predict(unseen_matrix)
display(pd.DataFrame({"unseen_description": unseen_texts, "predicted_cluster": predicted_clusters}))


In [ ]:
# Inspect cluster keywords and create a simple content-based recommendation function
from sklearn.metrics.pairwise import cosine_similarity

terms = tfidf.get_feature_names_out()
print("Top TF-IDF terms for each tuned K-Means cluster:")
for cluster_id, center in enumerate(best_kmeans.cluster_centers_):
    top_indices = center.argsort()[-10:][::-1]
    print(f"Cluster {cluster_id}: " + ", ".join(terms[top_indices]))

def recommend_titles(title, top_n=5):
    """Recommend titles with similar TF-IDF metadata to an exact title in the dataset."""
    title_col = "title"
    matches = model_df.index[model_df[title_col].fillna("").str.casefold() == str(title).casefold()].tolist()
    if not matches:
        available = model_df[title_col].dropna().head(10).tolist()
        raise ValueError(f"Title not found: {title!r}. Example titles: {available}")
    idx = matches[0]
    scores = cosine_similarity(X_tfidf[idx], X_tfidf).ravel()
    scores[idx] = -1  # exclude the selected title itself
    top_indices = np.argsort(scores)[::-1][:top_n]
    cols = [c for c in ["title", "type", "release_year", "listed_in", "description"] if c in model_df.columns]
    result = model_df.iloc[top_indices][cols].copy()
    result["similarity_score"] = scores[top_indices]
    return result.reset_index(drop=True)

# Example: replace this with a title that exists in your CSV
example_title = model_df["title"].dropna().iloc[0]
print("Recommendations for:", example_title)
display(recommend_titles(example_title, top_n=5))


### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

This project provides an end-to-end exploratory workflow for understanding Netflix catalogue metadata and grouping titles by textual similarity. It includes data cleaning, visualization, hypothesis testing, text normalization, TF-IDF feature extraction, dimensionality reduction, and a comparison of unsupervised clustering methods. The model evaluation uses silhouette scores and cluster-size inspection because no ground-truth labels are available. The resulting clusters and content-based similarity scores are prototypes only; real recommendation quality would need validation with user viewing and engagement data. After running the notebook, add two or three specific findings supported by the actual charts and p-values, and record the final model choice based on the measured results.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***